# 과학적 관리법 자율 문항 출제 에이전트 시스템

이 프로젝트는 강의 노트(PDF/TXT)를 분석하여 자율적으로 문제은행을 구축하고, 최종 학생용 문제지 및 교수용 해설지를 `.docx` 워드 파일로 생성하는 시스템입니다.

## 환경 설정 및 실행 방법
본 프로그램은 Google Colab 환경에서 실행하는 것을 권장합니다.

### 1. 필수 라이브러리 설치
Colab 내부에서 아래 명령어를 통해 필요한 패키지를 먼저 설치해 주세요.
!pip install google-genai pypdf python-docx requests wikipedia-api

### 2. Vertex AI 인증 및 프로젝트 세팅
GCP 크레딧을 적용하여 Gemini 모델을 호출하기 위해 Vertex AI 인증이 필요합니다.
1. 코드 상단의 auth.authenticate_user() 셀을 실행하여 구글 계정 로그인을 완료합니다.
2. 코드 내 PROJECT_ID 변수에 본인의 GCP 프로젝트 ID를 입력합니다.

# 1-1. Gemini API 환경 세팅 가이드


## 학습 목표
1. Google Cloud Platform (GCP) 신규 계정 생성 및 **$300 무료 크레딧** 활성화
2. Google Colab에서 GCP 계정으로 **Vertex AI 인증**
3. Google Gen AI SDK 및 Google ADK 설치 확인
4. Gemini API 첫 번째 호출 성공

---

> **주의사항:**
> - 신용카드/체크카드가 필요합니다 (실제 결제 없음, 본인 확인 용도)
> - 신규 계정에 한해 $300 크레딧(약 90일간 유효)이 제공됩니다
> - GCP 크레딧은 **Vertex AI 경로**에서만 적용됩니다 (AI Studio API 키 방식 불가)
> - 이 수업에서 사용하는 gemini-2.5-flash는 매우 저렴하므로 크레딧이 충분합니다

## 1. GCP 계정 생성 및 무료 크레딧 활성화

### 1.1 Google Cloud 계정 만들기

1. **[https://cloud.google.com](https://cloud.google.com)** 에 접속
2. 오른쪽 상단 **"무료로 시작하기"** 클릭
3. Google 계정으로 로그인 (학교 이메일이 아닌 @gmail.com 도메인 이메일이어야 함)
4. 국가 선택: **대한민국**
5. 서비스 약관 동의 체크

### 1.2 결제 정보 입력 (본인 확인용)

1. 계정 유형: **개인** 선택
2. 카드 정보 입력 (실제 결제 X, 본인 확인만)
3. **"무료 평가판 시작"** 클릭
4. **$300 크레딧 활성화 확인**

---

## 2. GCP 프로젝트 생성

### 2.1 새 프로젝트 만들기

1. GCP 콘솔 상단 프로젝트 선택 드롭다운 클릭
2. **"새 프로젝트"** 클릭
3. 프로젝트 이름: `agent-system-lab` (또는 원하는 이름)
4. **"만들기"** 클릭

### 2.2 Agent Platform API 활성화

1. 좌측 메뉴 → **"API 및 서비스"** → **"라이브러리"**
2. 검색창에 `Agent Platform API` 입력
3. **"Agent Platform API"** 선택 → **"사용"** 클릭
4. 잠시 기다리면 활성화 완료

> **참고:** API 이름이 "Vertex AI API" 또는 "AI Platform API"로 표시될 수도 있습니다.
> 서비스 이름(`aiplatform.googleapis.com`)이 맞으면 동일한 API입니다.

---

## 3. 프로젝트 ID 확인

1. GCP 콘솔 상단 프로젝트 선택 드롭다운 클릭
2. 현재 프로젝트의 **프로젝트 ID** (이름과 다를 수 있음) 복사
3. 아래 실습 코드의 `PROJECT_ID` 변수에 붙여넣기

## 4. Google Colab 환경 설정

### 4.1 Vertex AI 인증 (API 키 불필요)

**GCP 크레딧을 사용하려면 Vertex AI 경로로 인증해야 합니다.**
API 키 없이 Google 계정으로 직접 인증합니다.

아래 코드를 실행하면 **Google 계정 로그인 팝업**이 나타납니다:

In [ ]:
# Vertex AI 인증 (Google 계정으로 로그인)
from google.colab import auth

auth.authenticate_user()
print('✅ Google 계정 인증 성공!')
print('   이제 GCP 크레딧으로 Gemini API를 사용할 수 있습니다.')

✅ Google 계정 인증 성공!
   이제 GCP 크레딧으로 Gemini API를 사용할 수 있습니다.


## 5. 필수 라이브러리 설치

In [ ]:
# ① Google Gen AI SDK (기본 Gemini API 호출)
!pip install -q google-genai

# ② Google ADK (에이전트 개발 키트 - 3~4주차 사용)
!pip install -q google-adk

# ③ 기타 유용한 라이브러리
!pip install -q requests wikipedia-api

print('\n✅ 설치 완료!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 8.5 MB/s eta 0:00:00

✅ 설치 완료!


In [ ]:
# 설치된 버전 확인
from google import genai
from google.genai import types
import importlib.metadata

packages = ['google-genai', 'google-adk']
for pkg in packages:
    try:
        version = importlib.metadata.version(pkg)
        print(f'✅ {pkg}: v{version}')
    except:
        print(f'❌ {pkg}: 설치되지 않음')

✅ google-genai: v1.68.0
✅ google-adk: v1.29.0


## 6. 첫 번째 Gemini API 호출

모든 설정이 완료됐습니다! 첫 번째 AI 응답을 받아봅시다.
`PROJECT_ID` 변수에 GCP 콘솔에서 복사한 프로젝트 ID를 붙여넣기 하세요.

In [ ]:
from google import genai
from google.genai import types
from google.genai.types import HttpOptions
from google.colab import auth

auth.authenticate_user()

PROJECT_ID = "scimanagementhw2"
LOCATION = "us-central1"

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
    http_options=HttpOptions(api_version="v1"),
)

# 첫 번째 질문!
response = client.models.generate_content(model='gemini-2.5-flash', contents='AI 에이전트가 무엇인지 한국어로 2문장으로 설명해줘.')

print('🤖 Gemini의 답변:')
print(response.text)

🤖 Gemini의 답변:
AI 에이전트는 환경을 인지하고 목표를 달성하기 위해 행동하는 지능형 소프트웨어 또는 하드웨어 개체입니다. 이러한 에이전트는 환경으로부터 정보를 받아들여 스스로 판단하고, 정해진 목표를 향해 자율적으로 움직입니다.


## 7. 비용 모니터링 설정

### 예산 알림 설정 (권장)

1. GCP 콘솔 → **"결제"** → **"예산 및 알림"**
2. **"예산 만들기"** 클릭
3. 예산 금액: \$10 (안전한 수준)
4. 알림 비율: 50%, 90%, 100%
5. 이메일 알림 수신 설정

### 이 수업에서 사용하는 모델 비용 안내 (Vertex AI 기준)

| 모델 | 입력 (1M tokens) | 출력 (1M tokens) | 비고 |
|------|----------|----------|------|
| gemini-2.5-flash | \$0.15 | \$0.60 | 수업 기본 모델 |
| gemini-2.5-pro | \$1.25 | \$10.00 | 참고용 (거의 사용 안 함) |

> **예상 비용**: 4주 수업 전체 실습 기준 약 **\$2~5** 예상 (크레딧으로 충분)
>
> **\$300 크레딧은 Vertex AI 경로에서만 사용됩니다.**
> `auth.authenticate_user()` + `vertexai=True` 설정 시 크레딧이 자동 적용됩니다.

## 8. 환경 설정 완료 체크리스트

In [ ]:
# 최종 환경 점검
print('=' * 50)
print('🔍 환경 설정 최종 점검')
print('=' * 50)

checks = [
    ('GCP 계정 생성', True),
    ('$300 크레딧 활성화', True),
    ('Agent Platform API 활성화', True),
]

# 실제 확인 가능한 항목들
try:
    from google.colab import auth
    from google import genai
    from google.genai.types import HttpOptions
    _c = genai.Client(
        vertexai=True,
        project="scientific-management-494205",
        location="us-central1",
        http_options=HttpOptions(api_version="v1"),
    )
    checks.append(('Vertex AI 연결', True))
except Exception as e:
    checks.append(('Vertex AI 연결', False))

try:
    from google import genai
    from google.genai import types
    checks.append(('google-genai 설치', True))
except:
    checks.append(('google-genai 설치', False))

try:
    import google.adk
    checks.append(('google-adk 설치', True))
except:
    checks.append(('google-adk 설치', False))

# 결과 출력
all_passed = True
for name, status in checks:
    icon = '✅' if status else '❌'
    print(f'{icon} {name}')
    if not status:
        all_passed = False

print('=' * 50)
if all_passed:
    print('🎉 모든 설정 완료! 실습을 시작할 준비가 됐습니다!')
else:
    print('⚠️  일부 항목을 확인하세요. 위의 가이드를 다시 참고하세요.')

🔍 환경 설정 최종 점검


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


✅ GCP 계정 생성
✅ $300 크레딧 활성화
✅ Agent Platform API 활성화
✅ Vertex AI 연결
✅ google-genai 설치
✅ google-adk 설치
🎉 모든 설정 완료! 실습을 시작할 준비가 됐습니다!


---

## 참고 링크

| 자료 | 링크 |
|------|------|
| GCP 콘솔 | https://console.cloud.google.com |
| Vertex AI Gemini 문서 | https://cloud.google.com/vertex-ai/generative-ai/docs |
| Google Gen AI SDK | https://pypi.org/project/google-genai/ |
| Google ADK 문서 | https://google.github.io/adk-docs/ |
| GCP 무료 크레딧 | https://cloud.google.com/free |

## 자주 묻는 질문

**Q: 카드가 없으면 어떻게 하나요?**
A: Google AI Studio (aistudio.google.com) 에서는 카드 없이도 무료 API 키를 발급받을 수 있습니다. 단, GCP 크레딧은 사용할 수 없습니다.

**Q: 크레딧이 다 쓰이면 자동 결제되나요?**
A: 아니요. 자동으로 유료 계정으로 전환되지 않습니다. 직접 업그레이드해야 합니다.

**Q: gemini-2.5-flash 말고 다른 모델을 써도 되나요?**
A: 네! gemini-2.5-pro도 사용 가능합니다. 더 강력하지만 비용이 높습니다.

**Q: auth.authenticate_user() 팝업이 안 뜨면?**
A: Colab 설정 → 팝업 차단 해제, 또는 브라우저에서 직접 인증 링크를 열어 진행하세요.

In [ ]:
!pip install google-genai pypdf
## pypdf 라이브러리 설치 : pdf 파일 리더

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 3.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
folder_path = "/content/drive/MyDrive/scimgmt_hw2"
## Google Drive에 있는 강의 파일을 불러올 수 있도록 import

In [ ]:
import os
import glob
from pypdf import PdfReader

def load_all_lecture_notes_from_dir(dir_path):
    """
    지정한 폴더 경로 내에 존재하는 모든 PDF 및 TXT 파일을
    자동으로 검색하여 하나의 통합 지식 베이스(Knowledge Base)로 병합하는 도구
    """
    if not os.path.isdir(dir_path):
        print(f"❌ [도구 에러] '{dir_path}'는 올바른 폴더 경로가 아닙니다.")
        return ""

    print(f"📂 [Data Loader Tool] '{dir_path}' 폴더 내 강의 자료 탐색 시작...")

    # 폴더 내 모든 파일 중 .pdf와 .txt 파일만 추출 (대소문자 구분 없이)
    pdf_files = glob.glob(os.path.join(dir_path, "*.[pP][dD][fF]"))
    txt_files = glob.glob(os.path.join(dir_path, "*.[tT][xX][tT]"))
    all_files = sorted(pdf_files + txt_files) # 파일 이름 정렬해서 순서 맞추기

    if not all_files:
        print(f"⚠️ [도구 경고] 폴더 내에 지원하는 파일(.pdf, .txt)이 없습니다.")
        return ""

    print(f"🎯 총 {len(all_files)}개의 강의 자료를 발견했습니다.")

    merged_context = ""

    for idx, path in enumerate(all_files, 1):
        file_name = os.path.basename(path)
        ext = os.path.splitext(file_name)[1].lower()

        print(f"   ({idx}/{len(all_files)}) '{file_name}' 읽는 중...")

        # 1. PDF 처리
        if ext == ".pdf":
            try:
                reader = PdfReader(path)
                pdf_text = ""
                for page in reader.pages:
                    text = page.extract_text()
                    if text:
                        pdf_text += text + "\n"
                merged_context += f"\n\n=== [강의 데이터 소스 (PDF): {file_name}] ===\n"
                merged_context += pdf_text
            except Exception as e:
                print(f"   ❌ PDF 파싱 실패: {e}")

        # 2. TXT 처리
        elif ext == ".txt":
            try:
                with open(path, 'r', encoding='utf-8') as f:
                    content = f.read()
                merged_context += f"\n\n=== [강의 데이터 소스 (TXT): {file_name}] ===\n"
                merged_context += content
            except Exception as e:
                print(f"   ❌ TXT 로드 실패: {e}")

    return merged_context

In [ ]:
!pip install python-docx
## docx로 시험지 출력하도록 작성

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 3.0 MB/s eta 0:00:00


In [ ]:
# ==================================================================
# 💾 [공정 1] 단답/주관식 특화형 자율 검토 통합 출제기 에이전트 계층
# ==================================================================

## dir_path에 있는 모든 파일을 읽어서 하나로 합침
def load_all_lecture_notes_from_dir(dir_path):
    import os, glob
    from pypdf import PdfReader
    pdf_files = glob.glob(os.path.join(dir_path, "*.[pP][dD][fF]"))
    txt_files = glob.glob(os.path.join(dir_path, "*.[tTX][tT]"))
    all_files = sorted(pdf_files + txt_files)
    total_files = len(all_files)
    if not all_files: return ""
    print(f"📚 {total_files}개 파일 로딩 시작...")
    merged_context = ""
    for idx, path in enumerate(all_files, start=1):
        file_name = os.path.basename(path)
        print(f"   [{idx}/{total_files}] 로드 중: {file_name}")
        if path.lower().endswith(".pdf"):
            try:
                reader = PdfReader(path)
                for page in reader.pages:
                    text = page.extract_text()
                    if text: merged_context += text + "\n"
            except: pass
        else:
            try:
                with open(path, 'r', encoding='utf-8') as f:
                    merged_context += f.read() + "\n"
            except: pass
    return merged_context

def agent_advanced_bank_generator(lecture_note, short_count, essay_count, detailed_requirements):
    """[Phase 1-A] 단답형 및 주관식 2배수 문제은행 초안 생성"""      ### 주어진 문항수의 2배를 1차 문제은행으로 생성
    bank_short = short_count * 2
    bank_essay = essay_count * 2

    print(f"🤖 [Bank Generator] 1차 2배수 초안 풀(Pool) 구성 중...")

    system_instruction = f"""당신은 과학적관리 과목의 수석 출제위원입니다. 제공된 강의 데이터 소스를 바탕으로 아래 요구하는 수량의 문제은행 초안을 작성하십시오.

[출제 수량 제약조건]:
- 약술형 문제: 정확히 {bank_short}문제를 출제할 것. (핵심 개념 용어나 정의, 계산 결과를 간결하게 묻고 답하는 형태, 독립적으로 1번부터 {bank_short}번 부여)
- 주관식(서술/논술/계산형) 문제: 정확히 {bank_essay}문제를 출제할 것. (심도 있는 메커니즘 분석 및 계산 수식 활용 형태, 약술형에 이어서 {bank_short+1}번부터 {bank_essay}문제에 번호 부여)

[인간 검수자의 상세 지시]: {detailed_requirements}

반드시 다음 [좋은 문항 출제 기준]에 맞추어 문항을 구성해야 합니다.
[좋은 문제의 출제 기준]:
1. 개념의 본질 지향: 단순 지식의 파편만을 묻지 말고, 그 이론이나 메커니즘이 왜 등장했는지 본질과 배경에 대한 이해를 묻는 문제로 설계할 것.
2. 사고력 및 이해도 평가: 단순한 암기식 문항은 배제하고, 개인의 생각이나 비판적 시각을 요구하는 문항으로 설계할 것.
3. 적정 연산 복잡도: 손으로 계산할 수 있는 수준의 문항으로서, 필요 이상으로 계산이 꼬여서 핵심 개념 파악을 방해하는 문항은 지양할 것.

각 문항은 반드시 [문제 내용], [출제의도], [난이도 및 단원 정보], [모범 답안 및 상세 풀이]를 포함해야 합니다."""

    prompt = f"위 제약조건에 맞춰 최고 품질의 전문적인 문제은행 초안을 빌드하세요.\n\n[강의 노트]:\n{lecture_note}"
    response = client.models.generate_content(
        model='gemini-2.5-flash', contents=prompt, config=types.GenerateContentConfig(system_instruction=system_instruction)
    )
    return response.text

def agent_bank_reviewer(raw_bank, lecture_note, detailed_requirements):
    """[Phase 1-B] 성찰 패턴(Reflection) 기반 정정 검토 에이전트"""
    print("🔍 [Bank Reviewer] 생성된 문항 자율 검토 및 품질 튜닝 중...")
    system_instruction = (
        "당신은 과학적관리 과목의 '수석 문항 검토위원'입니다. 전 단계에서 생성된 2배수 문제은행 초안을 1차로 심사하십시오.\n\n"
        "🚨 [심사 및 정정 가이드라인]:\n"
        "1. 발문 심사: 문장이나 발문이 불필요하게 비대하거나 중언부언하는 경우 핵심 지식 위주로 정제하고 문장력을 명확하게 교정하십시오.\n"
        "2. 무결성 검증: 계산 문제의 수식과 수치 데이터가 강의노트 범위와 완벽히 일치하는지 검토하고, 답이 명확하게 떨어지는지 면밀히 검토하여 할루시네이션을 바로잡으십시오.\n"
        "3. 포맷 명확화: 인간 검수자가 최종 선별할 번호를 고르기 가장 편하도록 약술형 파트와 주관식 파트를 잘 구분하시오.\n"
        "4. 문항의 질 : [좋은 문제의 출제 기준] 3가지에 비추어, 문항의 수준을 검토하여 문항을 1차로 고도화하여 수정하시오.\n\n"
        "오직 검토와 교정이 끝난 '완벽한 품질의 2배수 문제은행 원안지 통합본'만 반환하십시오."
    )
    prompt = f"[인간 상세 지시사항]:\n{detailed_requirements}\n\n[1차 초안 문제은행]:\n{raw_bank}\n\n[검증용 강의노트]:\n{lecture_note}"
    response = client.models.generate_content(
        model='gemini-2.5-pro', contents=prompt, config=types.GenerateContentConfig(system_instruction=system_instruction)
    )
    return response.text

def agent_asymmetric_exam_builder(selected_cfg, edit_instructions, reviewed_bank, lecture_note):
    """[Phase 2] 인간 선택 반영 및 유연한 대체 선별 알고리즘이 내장된 최종 리파이너"""
    print("🤖 [Asymmetric Builder] 인간 선택 문항 수집, 주제 중복 자율 필터링 및 100점 만점 최적화 중...")
    system_instruction = (
        "당신은 최종 시험지 렌더링 자율 에이전트입니다. 검토 완료된 문제은행 원안에서 인간 검수자가 확정한 선택 번호를 기반으로 최종 시험지를 구축하십시오.\n\n"
        "🚨 [대체 선별 유연성 규칙]:\n"
        "- 인간 검수자가 특정 문제 번호들을 지정해 주었으나, 만약 선택된 문제들 간에 '주제나 핵심 개념이 심각하게 중복'된다고 판단되는 경우, "
        "지정된 번호 외에 문제은행 원안지 풀에 포함된 다른 고품질 예비 문항을 자율적으로 대체 선택하여 출제할 수 있는 전권을 부여합니다. 단, 최종 출제 수량은 엄격히 유지하십시오.\n\n"
        "⚠️ [최종 시험지 구성 지침]:\n"
        "1. [약술형 배치]: 선별된 단답형 문제들은 지면을 아끼기 위해 대문항 하나 아래에 배치하거나 전반부에 조밀하게 배치하십시오.\n"
        "2. [주관식 배치]: 서술형 및 계산형 주관식 문항들은 한 페이지에 한 문제씩 여유롭게 들어갈 수 있도록 각각 명확한 독립 대문항 번호(예: 문항 2, 문항 3...)를 부여하십시오.\n"
        "3. [100점 만점 스케일링]: 최종 완성된 모든 대문항들의 배점 합계가 정확하게 총합 '100점'이 되도록 난이도를 고려하여 공정하게 분배하십시오.\n"
        "4. 📌 중요: 출력물 본문 내에 반드시 명확하게 '---학생용 문제지---' 문자열 표식과 '---교수용 해설지---' 문자열 표식을 넣어 파트 구분을 완벽히 격리하십시오."
    )
    prompt = (
        f"[인간 검수자 최종 선택 및 대체 권한 정보]:\n{selected_cfg}\n\n"
        f"[추가 수정 및 편집 요청]:\n{edit_instructions}\n\n"
        f"[데이터 원천 - 검토 완료된 2배수 문제은행 원안지]:\n{reviewed_bank}\n\n"
        f"[강의자료 참조]:\n{lecture_note}"
    )
    response = client.models.generate_content(
        model='gemini-2.5-pro', contents=prompt, config=types.GenerateContentConfig(system_instruction=system_instruction)
    )
    return response.text

# ==================================================================
# 🖨️ [공정 2] 시험지 출력기
# ==================================================================

## 문제은행 (전체 문항) 출력기
def build_bank_docx(bank_text, filename):
    import os, re
    from docx import Document
    from docx.shared import Pt, Inches

    doc = Document()
    for s in doc.sections: s.top_margin = s.bottom_margin = s.left_margin = s.right_margin = Inches(1.0)
    style = doc.styles['Normal']
    style.font.name, style.font.size = 'Arial', Pt(11)
    style.paragraph_format.line_spacing = 1.25
    style.paragraph_format.space_after = Pt(8)

    doc.add_heading("중간고사 2배수 확장 문제은행 원안지", 0)
    lines = bank_text.split('\n')
    in_table = False
    table_data = []

    for line in lines:
        ls = line.strip()
        if not ls:
            if in_table and table_data:
                render_word_table(doc, table_data)
                table_data = []
                in_table = False
            continue

        if ls.startswith('|'):
            in_table = True
            if re.search(r'^[|\s:-]+$', ls): continue
            cells = [c.strip().replace("**", "") for c in ls.split('|')[1:-1]]
            table_data.append(cells)
            continue
        else:
            if in_table and table_data:
                render_word_table(doc, table_data)
                table_data = []
                in_table = False

        line_cleaned = ls.replace("**", "").replace("__", "").strip()
        p = doc.add_paragraph()
        if ls.startswith('#'):
            run = p.add_run(line_cleaned.replace('#','').strip())
            run.font.bold, run.font.size = True, Pt(14)
        elif any(line_cleaned.startswith(x) for x in ["•", "[", "정답", "출제의도", "문제", "문항", "난이도"]):
            p.add_run(line_cleaned).font.bold = True
        elif len(line_cleaned) > 0 and line_cleaned[0].isdigit() and ("." in line_cleaned or "번" in line_cleaned or ")" in line_cleaned):
            p.add_run(line_cleaned).font.bold = True
        else:
            p.add_run(line_cleaned)

    if in_table and table_data: render_word_table(doc, table_data)
    doc.save(filename)
    print(f"💾 문제은행 원안 워드 출력 완료 -> '{filename}'")


## 실제 최종 결과물 출력 엔진 (학생용 문제지 + 답안지)
def build_split_docx(final_text, filename_prefix):
    import os, re
    from docx import Document
    from docx.shared import Pt, Inches

    q_markers = ["---학생용 문제지---", "학생용 문제지", "---학생용 문제---", "[학생용 문제지]"]
    a_markers = ["---교수용 해설지---", "교수용 해설지", "---교수용 해설---", "[교수용 해설지]", "모범 답안", "정답 및 해설"]

    q_index = -1
    for m in q_markers:
        if m in final_text:
            q_index = final_text.find(m) + len(m)
            break

    a_index = -1
    for m in a_markers:
        if m in final_text:
            a_index = final_text.find(m)
            break

    if q_index == -1 or a_index == -1:
        half = len(final_text) // 2
        q_text = final_text[:half]
        a_text = final_text[half:]
    else:
        if q_index < a_index:
            q_text = final_text[q_index:a_index]
            a_text = final_text[a_index + 1:]
        else:
            a_text = final_text[a_index:q_index]
            q_text = final_text[q_index + 1:]

    def save_docx(text, title, filename, is_question_paper):
        doc = Document()
        for s in doc.sections: s.top_margin = s.bottom_margin = s.left_margin = s.right_margin = Inches(1.0)
        style = doc.styles['Normal']
        style.font.name, style.font.size = 'Arial', Pt(11)
        style.paragraph_format.line_spacing = 1.25
        style.paragraph_format.space_after = Pt(8)

        doc.add_heading(title, 0)
        is_first = True
        lines = text.split('\n')
        in_table = False
        table_data = []

        for line in lines:  ## 줄바꿈 처리기
            ls = line.strip()

            if any(tag in ls.lower() for tag in ["<br>", "<br/>", "<hr>", "<hr/>"]):
                continue
            if "다음 페이지에 계속" in ls:
                continue
            if ls == "---" or ls == "===":
                continue

            # [신규 필터 레이어] 문항 번호 옆이나 본문에 박힌 '[주관식 - 서술]', '[단답형]' 등의 대괄호 태그 제거
            # 배점 정보(예: [15점], [10점])는 남겨두기 위해 텍스트가 포함된 유형 성격의 대괄호만 정밀 타격합니다.
            ls = re.sub(r'\[(주관식|단답형|서술형|계산형|논술형|객관식|OX형)[^\]]*\]', '', ls).strip()

            if not ls:
                if in_table and table_data:
                    render_word_table(doc, table_data)
                    table_data = []
                    in_table = False
                continue

            if ls.startswith('|'):
                in_table = True
                if re.search(r'^[|\s:-]+$', ls): continue
                cells = [c.strip().replace("**", "") for c in ls.split('|')[1:-1]]
                table_data.append(cells)
                continue
            else:
                if in_table and table_data:
                    render_word_table(doc, table_data)
                    table_data = []
                    in_table = False

            line_cleaned = ls.replace("**", "").replace("__", "").strip()
            clean_ls = re.sub(r'[#\*_\s-]', '', line_cleaned)
            is_new_macro_q = clean_ls.startswith("문항") or clean_ls.startswith("문제")

            if is_question_paper and is_new_macro_q:
                if not is_first: doc.add_page_break()
                is_first = False

            p = doc.add_paragraph()
            if ls.startswith('#'):
                run = p.add_run(line_cleaned.replace('#','').strip())
                run.font.bold, run.font.size = True, Pt(14)
            elif is_new_macro_q or any(line_cleaned.startswith(x) for x in ["•", "[", "배점", "정답", "출제의도"]):
                p.add_run(line_cleaned).font.bold = True
            elif len(clean_ls) > 0 and clean_ls[0].isdigit() and ("." in line_cleaned or "번" in line_cleaned or ")" in line_cleaned):
                p.add_run(line_cleaned).font.bold = True
            else:
                p.add_run(line_cleaned)

        if in_table and table_data: render_word_table(doc, table_data)
        doc.save(filename)
        print(f"💾 독립 산출물 파일 동기화 완료 -> '{filename}'")

    save_docx(q_text, "2026학년도 제1학기 [과학적관리] 중간고사 문제지", f"{filename_prefix}_Questions.docx", True)
    save_docx(a_text, "2026학년도 제1학기 [과학적관리] 모범 답안 및 채점 해설서", f"{filename_prefix}_Answers.docx", False)

## 표 출력을 위한 데이터
def render_word_table(doc, table_data):
    if not table_data: return
    from docx.shared import Pt
    rows = len(table_data)
    cols = max(len(r) for r in table_data)
    table = doc.add_table(rows=rows, cols=cols)
    table.style = 'Light Shading Accent 1'
    for r_idx, row in enumerate(table_data):
        for c_idx, cell_value in enumerate(row):
            if c_idx < len(row):
                cell = table.cell(r_idx, c_idx)
                cell.text = cell_value
                if r_idx == 0:
                    for paragraph in cell.paragraphs:
                        for run in paragraph.runs: run.font.bold = True
    doc.add_paragraph().paragraph_format.space_after = Pt(12)

In [ ]:
## 문제은행 생성기

# 1. 강의 노트 소스 데이터 병합
target_folder = "/content/drive/MyDrive/scimgmt_hw2"
merged_notes = load_all_lecture_notes_from_dir(target_folder)

# 2. ✍️ 조별 회의 결과 반영 [최종 시험지] 타겟 문항 수 세팅
final_desired_short = 5    # 최종 시험지에 넣을 약술형 개수 (원안지는 x2개 자동 출제)
final_desired_essay = 5    # 최종 시험지에 넣을 주관식 개수 (원안지는 x2개 자동 출제)

# 3. 문제은행 출제 시 상세 가이드라인 지정 : 자유롭게 조정 가능함.
human_detailed_prompts = """
- 난이도 및 변별력: 변별력을 위해 약술형 원안지 중 2문항은 심도 있는 정의를 묻는 고난도(상)로 세팅하고, 주관식은 중~상 위주로 배치해줘.
- 계산 문제 제약: 주관식 원안지 중 최소 3문제 이상은 '생산 공정 라인 밸런싱(Line Balancing) 효율 계산' 및 '스톱워치 여유 시간(Allowance) 산정' 공식이 반영된 수치 계산형으로 출제해줘.
- 소문항 : 주관식 문제 중 일부는 핵심 개념에 대한 이해 및 개인의 생각을 묻는 구조의 소문항이 포함되도록 구성해줘.
"""

# 🚀 2배수 동적 생성 및 AI 자율 검토 검증 일괄 체인 가동
raw_bank_draft = agent_advanced_bank_generator(merged_notes, final_desired_short, final_desired_essay, human_detailed_prompts)
print("-" * 90)

final_reviewed_bank = agent_bank_reviewer(raw_bank_draft, merged_notes, human_detailed_prompts)
print("-" * 90)

# 4. 정제 완료된 2배수 문제은행 원안지 워드 출력
build_bank_docx(final_reviewed_bank, "HW2_Scientific_Management_Bank_Draft.docx")

ModuleNotFoundError: No module named 'pypdf'

In [ ]:
## 최종 시험지 및 예시답안 출력

# ------------------------------------------------------------------
# ✍️ 인간 검수자의 지시사항 입력 (OX 원천 차단 레이어 반영)
# ------------------------------------------------------------------

# 1. 사용할 문항 번호를 명시하세요.
selected_numbers = """
- 선택한 약술형 후보군: 1, 2, 4, 7, 8
- 선택한 주관식 후보군: 11, 12, 13, 15, 16, 17, 20
"""

# 2. OX 변형 출제 버그 배제 + 편집 지시사항 입력 + 배점 자동 입력. 특정 문항을 정해 수정 방향을 제시하는 것도 가능함.
specific_edits = (
    "🚨 [출제 유형 엄격 제한]: 최종 시험지의 약술형 문항들은 절대로 'O/X를 선택하는 문제'로 만들지 마십시오. "
    "오직 특정 용어, 핵심 개념 단어, 혹은 한 줄 이내의 핵심 문장을 직접 펜으로 작성해야 하는 '순수 주관식 약술형'으로만 구성해야 합니다. "
    "만약 원안지에 OX 형태로 가공된 문제가 있다면, 이를 즉시 '무엇이라 하는지 쓰시오' 형태의 순수 약술형 문항으로 문장 구조를 리팩토링하여 반영하십시오.\n\n"
    "- 추가 지시: 선택한 문항 중 주제가 완벽하게 겹치는 문제가 발견되면 원안지 풀(Pool) 내의 다른 번호로 자율 대체해도 좋습니다. "
    "단답형과 주관식 수량 비율을 정확히 유지하고, 전체 만점은 약술형 40점, 주관식 60점으로 총합 100점 만점이 되도록 난이도별 배점을 정교하게 밸런싱하십시오."
)

# ------------------------------------------------------------------
# 🚀 후공정 최종 리팩토링 및 강제 파일 덤프 세션 가동
# ------------------------------------------------------------------

final_exam_text = agent_asymmetric_exam_builder(
    selected_cfg=selected_numbers,
    edit_instructions=specific_edits,
    reviewed_bank=final_reviewed_bank,
    lecture_note=merged_notes
)

print("\n🤖 최종 정제 완료. 워드 파일 생성을 시작합니다...")
build_split_docx(final_exam_text, "HW2_Scientific_Management_Final")

🤖 [Asymmetric Builder] 인간 선택 문항 수집, 주제 중복 자율 필터링 및 100점 만점 최적화 중...

🤖 최종 정제 완료. 워드 파일 생성을 시작합니다...
💾 독립 산출물 파일 동기화 완료 -> 'HW2_Scientific_Management_Final_Questions.docx'
💾 독립 산출물 파일 동기화 완료 -> 'HW2_Scientific_Management_Final_Answers.docx'


### legacy

!!! 전체 파이프라인

In [ ]:
# ==================================================================
# 💾 [공정 1] 출제기 핵심 에이전트 계층 및 파일 로더
# ==================================================================

def load_all_lecture_notes_from_dir(dir_path):
    if not os.path.isdir(dir_path):
        print(f"❌ [도구 에러] '{dir_path}'는 올바른 폴더 경로가 아닙니다.")
        return ""
    pdf_files = glob.glob(os.path.join(dir_path, "*.[pP][dD][fF]"))
    txt_files = glob.glob(os.path.join(dir_path, "*.[tT][xX][tT]"))
    all_files = sorted(pdf_files + txt_files)

    total_files = len(all_files)
    if not all_files:
        print("ℹ️ 폴더 내에 인식 가능한 강의자료 파일(PDF, TXT)이 없습니다.")
        return ""

    print(f"📚 총 {total_files}개의 강의자료 파일을 검색했습니다. 로딩을 시작합니다...")

    merged_context = ""
    for idx, path in enumerate(all_files, start=1):
        file_name = os.path.basename(path)
        ext = os.path.splitext(file_name)[1].lower()

        print(f"   [{idx}/{total_files}] 로드 중: {file_name}")

        if ext == ".pdf":
            try:
                reader = PdfReader(path)
                for page in reader.pages:
                    text = page.extract_text()
                    if text: merged_context += text + "\n"
            except Exception as e:
                print(f"   ⚠️ {file_name} PDF 읽기 실패: {e}")
        elif ext == ".txt":
            try:
                with open(path, 'r', encoding='utf-8') as f:
                    merged_context += f.read() + "\n"
            except Exception as e:
                print(f"   ⚠️ {file_name} TXT 읽기 실패: {e}")

    return merged_context

class BaseAgentWorker:
    def __init__(self, name: str, task_id: str):
        self.name = name
        self.task_id = task_id

class ContextAnalyzerAgent(BaseAgentWorker):
    def __init__(self): super().__init__("Context Analyzer", "Task_0")
    def run(self, lecture_note):
        print(f"🤖 [{self.name}] 원본 강의 데이터 맥락 분석 및 핵심 키워드 식별 시작...")
        prompt = f"다음 강의 데이터를 분석하여 핵심 출제 키워드와 단원별 학습 목표를 만들어 :\n\n{lecture_note}"   ## 학습목표, 예시 등 먼저 찾아보게 하면 어떨까
        response = client.models.generate_content(model='gemini-2.5-flash', contents=prompt)
        print(f"   -> [{self.name}] 분석 완료.")
        return response.text

class ExamExecutorAgent(BaseAgentWorker):
    def __init__(self): super().__init__("Exam Executor", "Task_2")
    def run(self, lecture_note, exam_plan):
        print(f"🤖 [{self.name}] 출제 계획 가이드라인 기반 20문항 대량 문제은행 풀(Pool) 집필 시작...")  ## 단원별로 몇 문제 출제할지를 처음에 입력할 때 줄 수 있는 방안을 찾기. (프롬포트에 반영?)
        prompt = f"전 단계에서 작성된 출제 계획서를 바탕으로, 제공된 강의 데이터 소스에서 중간고사 시험 문제와 그에 대응하는 모범 답안을 집필하십시오. 계획서의 문항 수, 배점, 난이도 밸런스를 엄격히 준수해야 합니다.:\n\n출제 계획서:\n{exam_plan}\n\n강의 데이터 소스:\n{lecture_note}"
        response = client.models.generate_content(model='gemini-2.5-flash', contents=prompt)
        print(f"   -> [{self.name}] 20문항 초안 집필 완료.")
        return response.text

def agent_context_analyzer(lecture_note: str) -> str:
    return ContextAnalyzerAgent().run(lecture_note)

def agent_planner(lecture_note: str, core_concepts: str, prompt_requirements: str) -> str:
    print("🤖 [Blueprint Planner] 출제 전략 및 구조 설계 시작...")
    system_instruction = (
        "당신은 과학적관리 과목의 '수석 출제위원'입니다. "
        "아래의 [좋은 문제의 출제 기준]을 엄격히 준수하여 20개의 대량 문제 후보군(Pool)을 발굴하기 위한 계획서를 설계하세요.\n\n"
        "[좋은 문제의 출제 기준]:\n"
        "1. 개념의 본질 지향: 단순 지식의 파편을 묻지 말고, 그 이론이나 메커니즘이 왜 등장했는지 본질과 배경을 규명할 것.\n"
        "2. 사고력 및 이해도 평가: 암기식 문항은 배제하고, 개인의 생각이나 비판적 시각을 요구하는 문항으로 설계할 것.\n"
        "3. 적정 연산 복잡도: 불필요하게 계산이 꼬여서 핵심 개념 파악을 방해하는 문항은 전면 지양할 것."
            ## 문제은행에서 10문제 선별하는 과정은 인간이 하도록 option 넣기.
            ## 1차로 20문제짜리(AI가 1차 검토한) 시험지 파일을 받고, 1~20번 중에서 사용할 문제 번호만 입력하면 AI가 거기에 맞춰서 시험지 구성 + 예시답안 작성한 워드파일 출력하도록 구현
            ## 고르는 거 + 내고싶은 문제...? 아이디어 중간에 추가할 기회를 만들자... 정도 -> 수정 프롬포트를 중간 기회로 열어두기. 번호입력+특정문항수정...을 하나의 프롬포트로 (만들면서 효율적인 방식 적당히)
            ## 구성은 다시 AI가 해줄 수 있도록 프롬포트를 짜기.
            ## 수정하는 단계가 어렵다면, 수정할 문제는 인간이 수정할 수 있도록... 알잘딱 --> '최종 구성 프롬포트'를 적당히 만들면 될 듯.
    )
    prompt = f"[요구사항]: {prompt_requirements}\n[핵심지식]: {core_concepts}\n[전체데이터]: {lecture_note}"
    response = client.models.generate_content(
        model='gemini-2.5-flash', contents=prompt, config=types.GenerateContentConfig(system_instruction=system_instruction)
    )
    print("   -> [Blueprint Planner] 가이드라인 수립 완료.")
    return response.text

def agent_executor(lecture_note: str, exam_plan: str) -> str:
    return ExamExecutorAgent().run(lecture_note, exam_plan)

def agent_reviewer(lecture_note: str, prompt_requirements: str, exam_draft: str) -> str:
    print("🤖 [Quality Reviewer] 루브릭 기반 최우수 10문항 자율 선별 시작...")
    system_instruction = (
        "당신은 시험 문항을 최종 심사 및 선별하는 '수석 검토위원'입니다.\n"
        "전 단계에서 생성된 20개의 후보군 중 가장 완성도가 높은 최우수 10문항을 선별하십시오.\n\n"  ## 검수 절차 추가하기
        "⚠️ [엄격한 출력 포맷 제약조건]:\n"
        "1. 당신은 오직 '학생들이 보는 순수한 시험 문제지 내용'만 출력해야 합니다.\n"
        "2. 어느 단원에서 출제되었는지, 왜 선별했는지 등 '교수/조교용 부연설명'은 절대로 문제지에 적지 마십시오. (그 정보는 다음 에이전트가 처리합니다)\n"
        "3. 가독성을 위해 문항 번호, 문제 본문, (있다면) 보기나 지문 사이에는 반드시 명확하게 줄바꿈(Enter)을 넣어서 단락을 분리하십시오.\n"
        "아래 <출력 예시>를 똑같이 따를 필요는 없으나, 모든 문항에서 통일성을 유지해야 하며, 종결 어미는 '~하십시오.' 대신 '하시오.' '~입니다.' 대신 '~이다.' 와 같이 통일하십시오.\n\n"
        "출력 예시:\n"
        "## [제1부] 서술형 문항\n"
        "문항 1. 테일러의 과학적 관리법 원칙에 대해 설명하시오. [10점]\n\n"
        "문항 2. 다음 수식을 활용하여 최적 작업량을 계산하시오. [15점]"   ## 마지막에 사람이 한 번 검수하는 절차 추가?
    )
    prompt = f"[사용자 요구사항]: {prompt_requirements}\n[후보문항들]: {exam_draft}\n[원본노트]: {lecture_note}"
    response = client.models.generate_content(
        model='gemini-2.5-pro', contents=prompt, config=types.GenerateContentConfig(system_instruction=system_instruction)
    )
    print("   -> [Quality Reviewer] 학생 배포용 최종 10문항 확정 완료.")
    return response.text

def agent_solution_architect(final_questions: str, lecture_note: str) -> str:
    print("🤖 [Solution Architect] 공식 모범 답안 및 채점 기준표(Rubric) 집필 시작...")
    system_instruction = (
        "당신은 '수석 채점 시스템 설계자'입니다.\n"
        "넘겨받은 [확정된 중간고사 10문항] 각각에 대하여 다음 3가지 요소를 세트로 묶어 해설서를 작성하십시오.\n\n"
        "1. [출제 정보]: 해당 문제가 강의노트의 '어느 단원(대주제/소주제)'에서 출제된 것인지 명시하고, 출제 의도에 대한 부연설명을 기술할 것.\n"
        "2. [모범 답안]: 핵심 키워드가 포함된 가이드라인급 정답 제시.\n"
        "3. [채점 루브릭]: 부분점수 기준이 명시된 정량적 채점 기준.\n\n"
        "문항 간 구분이 잘 되도록 줄바꿈을 명확히 하고, 원본 강의 데이터에 기반하여 할루시네이션 없이 작성하십시오."
    )
    prompt = f"[확정된 10문항]:\n{final_questions}\n\n[검증용 원본 강의 데이터]:\n{lecture_note}"
    response = client.models.generate_content(
        model='gemini-2.5-flash', contents=prompt, config=types.GenerateContentConfig(system_instruction=system_instruction)
    )
    print("   -> [Solution Architect] 정답 해설서 공식 빌드 완료.")
    return response.text


# ==================================================================
# 🖨️ [공정 2] 소문항 무결성 보존형 표 렌더러 및 사족 필터 (최종 수정판)
# ==================================================================

def build_single_target_docx(title_text, text_content, filename, folder_path=None):
    if not text_content:
        print(f"❌ [빌드 실패] {filename}용 데이터가 비어있습니다.")
        return None

    doc = Document()

    for section in doc.sections:
        section.top_margin = Inches(1.0)
        section.bottom_margin = Inches(1.0)
        section.left_margin = Inches(1.0)
        section.right_margin = Inches(1.0)

    style = doc.styles['Normal']
    style.font.name = 'Arial'
    style.font.size = Pt(11)
    style.paragraph_format.line_spacing = 1.25
    style.paragraph_format.space_after = Pt(8)

    title_p = doc.add_paragraph()
    run_t = title_p.add_run(title_text)
    run_t.font.size = Pt(16)
    run_t.font.bold = True
    title_p.paragraph_format.space_after = Pt(12)

    meta_p = doc.add_paragraph()
    meta_run = meta_p.add_run("본 산출물은 공정 데이터 렌더러와 클렌징 필터가 결합된 무결성 결과물입니다.")
    meta_run.font.size = Pt(9)
    meta_p.paragraph_format.space_after = Pt(24)

    clean_text = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]', '', text_content)
    lines = clean_text.split('\n')

    is_first_question = True
    in_table = False
    table_data = []

    for line in lines:
        line_stripped = line.strip()
        if not line_stripped:
            if in_table and table_data:
                render_word_table(doc, table_data)
                table_data = []
                in_table = False
            continue

        # 🚨 [필터 리팩토링] 소문항 오인 삭제 전면 방지
        # 문제지(Questions)에서 오직 진짜 사족 라인만 정밀 타격하여 드롭
        if "Questions" in filename:
            # 명백한 단원 명시 정보나 메타데이터 성격의 헤더 행만 필터링
            if line_stripped.startswith('*') and any(k in line_stripped for k in ["단원", "출제", "모듈", "Module"]):
                continue
            if line_stripped.startswith("출제 정보") or line_stripped.startswith("부연설명"):
                continue

        # 🚨 마크다운 표 스트림 처리
        if line_stripped.startswith('|'):
            in_table = True
            if re.search(r'^[|\s:-]+$', line_stripped):
                continue
            cells = [c.strip().replace("**", "") for c in line_stripped.split('|')[1:-1]]
            table_data.append(cells)
            continue
        else:
            if in_table and table_data:
                render_word_table(doc, table_data)
                table_data = []
                in_table = False

        line_cleaned = line_stripped.replace("**", "").replace("__", "").strip()
        clean_prefix = re.sub(r'[#\*_\s-]', '', line_cleaned)

        # 새 대문항 시작 식별 정규식 (소문항 1., 2. 등은 페이지 분리 트리거에서 제외)
        is_new_question_line = (
            clean_prefix.startswith("문항") or
            clean_prefix.startswith("문제") or
            (line_cleaned.startswith("<문제") and line_cleaned.endswith(">"))
        )

        if "Questions" in filename and is_new_question_line:
            if not is_first_question:
                doc.add_page_break()
            is_first_question = False

        p = doc.add_paragraph()

        if line_stripped.startswith("###") or line_stripped.startswith("##") or line_stripped.startswith("#"):
            run = p.add_run(line_cleaned.replace("#", "").strip())
            run.font.bold = True
            run.font.size = Pt(13)
            p.paragraph_format.space_before = Pt(12)
        elif is_new_question_line or any(line_cleaned.startswith(prefix) for prefix in ["•", "-", "*", "[", "배점"]):
            run = p.add_run(line_cleaned)
            run.font.bold = True
        # ⭐️ 소문항 숫자 가독성 강조 레이어로 안전하게 포매팅
        elif len(clean_prefix) > 0 and clean_prefix[0].isdigit() and ("." in line_cleaned or "번" in line_cleaned or ")" in line_cleaned):
            run = p.add_run(line_cleaned)
            run.font.bold = True
        else:
            p.add_run(line_cleaned)

    if in_table and table_data:
        render_word_table(doc, table_data)

    if os.path.exists(filename):
        try: os.remove(filename)
        except: pass
    doc.save(filename)
    return filename

def render_word_table(doc, table_data):
    if not table_data: return
    rows = len(table_data)
    cols = max(len(r) for r in table_data)

    table = doc.add_table(rows=rows, cols=cols)
    table.style = 'Light Shading Accent 1'

    for r_idx, row in enumerate(table_data):
        for c_idx, cell_value in enumerate(row):
            if c_idx < len(row):
                cell = table.cell(r_idx, c_idx)
                cell.text = cell_value
                if r_idx == 0:
                    for paragraph in cell.paragraphs:
                        for run in paragraph.runs:
                            run.font.bold = True
    doc.add_paragraph().paragraph_format.space_after = Pt(12)

print("✅ [공정 1 & 2] 소문항 보존 강화형 표 생성 엔진 로드 완료!")

✅ [공정 1 & 2] 소문항 보존 강화형 표 생성 엔진 로드 완료!


In [ ]:
# ==================================================================
# ⛓️ [공정 3] 오케스트레이션 파이프라인 제어 계층 선언
# ==================================================================

def run_agentic_exam_generator_from_dir(folder_path, prompt_requirements):
    print("==========================================================================================")
    print("=== 🚀 Agentic Work System: 오리지널 프롬프트 기반 시험 출제 파이프라인 가동 ===")
    print("==========================================================================================")

    # Step 0. 데이터 소스 통합
    merged_notes = load_all_lecture_notes_from_dir(folder_path)
    if not merged_notes.strip():
        print("❌ [오케스트레이션 에러]: 강의 소스 파일이 존재하지 않습니다.")
        return None, None
    print(f"✅ [Data Pipeline] 단일 지식 컨텍스트 병합 완료 (크기: {len(merged_notes)} 자)")
    print("-" * 90)

    # Step 1. 지식 분석
    core_concepts = agent_context_analyzer(merged_notes)
    print("-" * 90)

    # Step 2. 블루프린트 수립 (Planner)
    exam_plan = agent_planner(merged_notes, core_concepts, prompt_requirements)
    print("-" * 90)

    # Step 3. 20문항 대량 문제은행 풀 구축 (Executor)
    exam_draft_20 = agent_executor(merged_notes, exam_plan)
    print("-" * 90)

    # Step 4. 3대 기준 기반 '순수 문제 10문항' 엄선 (Reviewer)
    clean_questions_10 = agent_reviewer(merged_notes, prompt_requirements, exam_draft_20)
    print("-" * 90)

    # Step 5. 10문항 전용 '모범 답안 및 출제의도' 집중 집필 (Solution Architect)
    model_answers_10 = agent_solution_architect(clean_questions_10, merged_notes)
    print("-" * 90)

    print("💾 시스템 독립 빌더 계층 가동: 2개의 독립된 워드 파일 출력을 시작합니다...")

    # 📌 빌더 함수 호출 시 매개변수 구조 동기화 (folder_path 전달 에러 완벽 해결)
    exam_file = build_single_target_docx(
        title_text="2026학년도 제1학기 중간고사 [과학적관리] 중간고사 문제지",
        text_content=clean_questions_10,
        filename="HW2_Scientific_Management_Questions.docx",
        folder_path=folder_path
    )

    answer_file = build_single_target_docx(
        title_text="2026학년도 제1학기 중간고사 [과학적관리] 모범 답안 및 출제의도 해설서",
        text_content=model_answers_10,
        filename="HW2_Scientific_Management_Answers.docx",
        folder_path=folder_path
    )

    print(f"🎉 [과제 최적화 완수] 시스템 오케스트레이션 공정이 성공적으로 종료되었습니다.")
    print(f"💾 [독립 생성] 학생 배포용 시험 문제지 워드 빌드 성공 -> '{exam_file}'")
    print(f"💾 [독립 생성] 교수/조교용 채점 가이드라인 워드 빌드 성공 -> '{answer_file}'")
    print("==========================================================================================")

    return clean_questions_10, model_answers_10

print("✅ [공정 3] 파이프라인 관리 함수 등록 완료!")

✅ [공정 3] 파이프라인 관리 함수 등록 완료!


In [ ]:
# ==================================================================
# 🏃‍♂️ [공정 4] 최종 메인 실행 코드
# ==================================================================

# 1. 원본 강의 노트가 들어있는 드라이브 폴더 경로 지정
target_folder = "/content/drive/MyDrive/scimgmt_hw2"

# 2. ipynb 원본 요구사항 문자열 그대로 지정
user_requirements = "준 파일 범위에서 난이도 중, 상을 섞어서 서술형, 논술형 문제 10개 출제해 줘. 각 출제 문제에 대해서는 어느 단원으로부터 나온 건지 부연설명을 해 주고. 계산 문제도 2문제 이상 포함되어야 해."
## ox 몇문제 / 객관식 몇문제 / 논술형 몇문제...

# 3. 🚀 파이프라인 원버튼 가동!
# 이 셀만 실행하면 상단의 공정 1, 2, 3이 순차적으로 절차에 맞게 호출되면서 2개의 분리된 무결성 워드가 깨끗하게 생성됩니다.
final_q_text, final_a_text = run_agentic_exam_generator_from_dir(target_folder, user_requirements)

=== 🚀 Agentic Work System: 오리지널 프롬프트 기반 시험 출제 파이프라인 가동 ===
📚 총 9개의 강의자료 파일을 검색했습니다. 로딩을 시작합니다...
   [1/9] 로드 중: Line Balancing Example Problems.pdf
   [2/9] 로드 중: M3 Supplement Digital Twin.pdf
   [3/9] 로드 중: M3.1.1 Micro-level Motion Study (Therbligs)_041626.pdf
   [4/9] 로드 중: M3.1.2 Process-level Motion Study (Process Charts)_041726.pdf
   [5/9] 로드 중: M3.1.3 Time Study_042326.pdf
   [6/9] 로드 중: M3.2 Worker-Machine Systems_043026.pdf
   [7/9] 로드 중: M3.3 Flow Production Work Systems_051426.pdf
   [8/9] 로드 중: MTM-1_data_card_EN.pdf
   [9/9] 로드 중: Sample Size Determination (Stopwatch Time Study).pdf
✅ [Data Pipeline] 단일 지식 컨텍스트 병합 완료 (크기: 105038 자)
------------------------------------------------------------------------------------------
🤖 [Context Analyzer] 원본 강의 데이터 맥락 분석 및 핵심 키워드 식별 시작...
   -> [Context Analyzer] 분석 완료.
------------------------------------------------------------------------------------------
🤖 [Blueprint Planner] 출제 전략 및 구조 설계 시작...
   -> [Blueprint Planner] 가이드라인 수립

KeyboardInterrupt: 